# Report figures

Rebuilds a pipeline HTML view with a **white background**, so it can be used in
the report. Everything else is the same as the file the pipeline writes: same
traces, same palette, same hover data, same axes.

The only change is `template='plotly_dark'` &rarr; `template='plotly_white'`.

Output goes next to this notebook, so the `.tex` files can find it.

In [9]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.colors as pcolors
import plotly.graph_objects as go
import plotly.io as pio

FIG_DIR = Path.cwd()                                   # report/Content/Figures
REPO = FIG_DIR.parents[2]                              # stentFIT/
OUT = REPO / "examples" / "data" / "output" / "stent_skeleton"

# STENT = "stent01"
# STENT = "stent01"
STENT = "stent01"
SRC = OUT / STENT

print("source:", SRC, "|", "found" if SRC.exists() else "MISSING")


# --- how the stent is framed in the exported PNGs -------------------------
# z is the stent axis and plotly puts z up, so this looks along x/y with a
# slight elevation: the stent stands vertically in the image.
#
# CAMERA_DIST is the knob to turn if the stent is cropped. With
# aspectmode='data' a stent is ~6x longer than wide, so the scene box is tall
# and the camera has to sit well back to fit all of it in. Larger = more
# zoomed out.
CAMERA_DIR = dict(x=1.0, y=1.0, z=0.25)   # viewing direction
CAMERA_DIST = 4.5                          # 3.0 crops the ends, 6.0 is too small
# NOTE: orthographic ignores CAMERA_DIST for zoom, so the stent stays cropped
# no matter how far back the camera goes. Keep this False.
ORTHOGRAPHIC = False

_n = math.sqrt(sum(v * v for v in CAMERA_DIR.values()))
CAMERA_EYE = {k: v / _n * CAMERA_DIST for k, v in CAMERA_DIR.items()}

PNG_W, PNG_H = 700, 1800     # tall canvas to match the standing stent


source: /Users/vural/Desktop/RWTH - SiSc/Projects/Github/stentFIT/examples/data/output/stent_skeleton/stent01 | found


## `ring_assignment` on white

Mirrors `stentfit.core.plotting.plot_points_3d_html(..., categorical=True)`, which
is what writes `ring_assignment.html` into the output folder.

In [ ]:
MAX_DISPLAY = 500_000     # same number of points the pipeline's view shows
POINT_SIZE = 1
SHOW_LEGEND = True        # False drops the ring_id legend on the right

df = pd.read_csv(SRC / "ring_points.csv")

disp = df.sample(MAX_DISPLAY, random_state=0) if len(df) > MAX_DISPLAY else df
note = f"  ({len(disp):,}/{len(df):,} shown)" if len(disp) < len(df) else ""

palette = pcolors.qualitative.Dark24 + pcolors.qualitative.Light24
labels = sorted(disp["ring_id"].unique())

fig = go.Figure()
for i, lab in enumerate(labels):
    sub = disp[disp["ring_id"] == lab]
    cdat = np.column_stack([sub["point_id"].to_numpy(), sub["ring_id"].to_numpy()])
    fig.add_trace(go.Scatter3d(
        x=sub["x"], y=sub["y"], z=sub["z"], mode="markers",
        marker=dict(size=POINT_SIZE, color=palette[i % len(palette)]),
        name=f"ring_id={lab}", customdata=cdat,
        hovertemplate=("point_id=%{customdata[0]}<br>"
                       "ring_id=%{customdata[1]}<extra></extra>")))

fig.update_layout(
    template="plotly_white",                      # <- the only change
    height=800, margin=dict(l=0, r=0, t=40, b=0),
    title=f"{STENT} rings ({len(labels)}){note}",
    showlegend=SHOW_LEGEND,
    legend=dict(itemsizing="constant"),
    scene=dict(aspectmode="data", xaxis_title="x", yaxis_title="y", zaxis_title="z",
               camera=dict(eye=CAMERA_EYE, up=dict(x=0, y=0, z=1),
                           projection=dict(
                               type="orthographic" if ORTHOGRAPHIC else "perspective"))))

out = FIG_DIR / "methodology_figures" / "ring_assignment_white.html"
pio.write_html(fig, out, auto_open=False)
print("wrote", out.name)

# static PNG for the report (needs: pip install --upgrade kaleido)
png = FIG_DIR / "methodology_figures" / "po_rings.png"
fig.write_image(png, width=PNG_W, height=PNG_H, scale=2)
print("wrote", png.name)

fig.show()

## `splines` on white

Mirrors `stentfit.core.plotting.plot_splines_html`, which writes `splines.html`
in the output folder. The fitted curves are rebuilt from `skeleton_splines.json`
rather than re-fitted: the JSON stores `degree`, `knot_vector` and
`control_points`, which is exactly scipy's `tck`, so the drawn curves are the
same ones the pipeline produced.

Again the only change to the plot is `plotly_dark` &rarr; `plotly_white`.

In [ ]:
import json

import matplotlib.pyplot as plt
from scipy.interpolate import splev

N_EVAL = 100          # points per curve, same default as the pipeline
LINE_WIDTH = 5

rec = json.loads((SRC / "skeleton_splines.json").read_text())
curves = rec["curves"]

cmap = plt.get_cmap("tab20")
palette = [f"rgb({int(r*255)},{int(g*255)},{int(b*255)})"
           for r, g, b, _ in (cmap(i % 20) for i in range(len(curves)))]

fig = go.Figure()
n_drawn = 0
for i, (spl, color) in enumerate(zip(curves, palette), start=1):
    if spl is None:
        continue
    ctrl = np.asarray(spl["control_points"])
    if spl["knot_vector"] is None:          # polyline fallback
        sp = ctrl
    else:
        tck = (np.asarray(spl["knot_vector"]), list(ctrl.T), int(spl["degree"]))
        uu = np.linspace(0.0, 1.0, N_EVAL)
        x, y, z = splev(uu, tck)
        sp = np.column_stack([x, y, z])
    fig.add_trace(go.Scatter3d(
        x=sp[:, 0], y=sp[:, 1], z=sp[:, 2], mode="lines",
        line=dict(width=LINE_WIDTH, color=color),
        name=f"curve {i}", hoverinfo="name", showlegend=False))
    n_drawn += 1

fig.update_layout(
    template="plotly_white",                  # <- the only change
    height=800, margin=dict(l=0, r=0, t=40, b=0),
    title=f"{n_drawn} fitted spline curves",
    scene=dict(aspectmode="data", xaxis_title="x", yaxis_title="y", zaxis_title="z",
               camera=dict(eye=CAMERA_EYE, up=dict(x=0, y=0, z=1),
                           projection=dict(
                               type="orthographic" if ORTHOGRAPHIC else "perspective"))))

out = FIG_DIR / "methodology_figures" / "splines_white.html"
pio.write_html(fig, out, auto_open=False)
print("wrote", out.name)

# static PNG for the report (needs: pip install --upgrade kaleido)
png = FIG_DIR / "methodology_figures" / "po_3d_splines.png"
fig.write_image(png, width=PNG_W, height=PNG_H, scale=2)
print("wrote", png.name)

fig.show()

## Why the 3D region adjacency is needed

Schematic for `fig:region-adjacency` in `03_methods.tex`. Two crowns face each other
with a narrow gap, as they do on a real stent. The gap survives in the point cloud,
but the dilation that is needed to close the speckle of a sampled cloud also closes
the gap, and the thinning then draws a strut that does not exist.

The geometry is synthetic, so that the effect is visible at a readable scale. The
`dilation` / `closing` / `skeletonize` calls and the nearest-region labelling are the
same ones `compute_skeleton_2d` and `check_skeleton_quality` use.

Writes `region_adjacency.png` next to this notebook.

In [ ]:
from matplotlib.lines import Line2D
from scipy.spatial import cKDTree
from skimage.morphology import skeletonize, dilation, closing, disk

OUT = FIG_DIR / "methodology_figures" / "region_adjacency.png"

W       = 0.10      # strut width
PX      = W / 10.0  # pixel size  ->  10 px across one strut
DILATE  = 4         # dilation radius, in pixels
RNG     = np.random.default_rng(3)

C_A    = "#3a6ea5"   # region A
C_B    = "#c98a2b"   # region B
C_BAD  = "#8c1923"   # flagged connection (report accent)
C_SOLID = "#b9c6d4"
C_SKEL  = "#2b2b2b"


def strut_points(poly, n_per_mm=9000):
    """Scatter points over a strut of width W following a polyline."""
    poly = np.asarray(poly, float)
    seg  = np.diff(poly, axis=0)
    ln   = np.linalg.norm(seg, axis=1)
    out  = []
    for p0, d, L in zip(poly[:-1], seg, ln):
        n  = int(n_per_mm * L)
        t  = RNG.random(n)[:, None]
        u  = d / L
        nv = np.array([-u[1], u[0]])
        s  = (RNG.random(n)[:, None] - 0.5) * W
        out.append(p0 + t * d + s * nv)
    # round the corners off so the crown looks like a crown
    for p in poly[1:-1]:
        n = int(n_per_mm * W * 0.55)
        a = RNG.random(n) * 2 * np.pi
        r = np.sqrt(RNG.random(n)) * W / 2
        out.append(p + np.column_stack([r * np.cos(a), r * np.sin(a)]))
    return np.vstack(out)


# two crowns facing each other, tip to tip, with a narrow gap between them
A = strut_points([(0.00, 0.30), (0.52, 0.55), (0.00, 0.80)])
B = strut_points([(1.30, 0.30), (0.70, 0.55), (1.30, 0.80)])

pts = np.vstack([A, B])
reg = np.r_[np.zeros(len(A), int), np.ones(len(B), int)]

# ---- rasterise, dilate, close, thin (same operations as compute_skeleton_2d) ----
lo   = pts.min(0) - 6 * PX
n_c  = int(np.ceil((pts[:, 0].max() + 6 * PX - lo[0]) / PX))
n_r  = int(np.ceil((pts[:, 1].max() + 6 * PX - lo[1]) / PX))
col  = np.clip(((pts[:, 0] - lo[0]) / PX).astype(int), 0, n_c - 1)
row  = np.clip(((pts[:, 1] - lo[1]) / PX).astype(int), 0, n_r - 1)

img = np.zeros((n_r, n_c), bool)
img[row, col] = True
solid = closing(dilation(img, footprint=disk(DILATE)), footprint=disk(1))
skel  = skeletonize(solid)

# ---- label every skeleton pixel by its nearest surface point's region ----
sr, sc  = np.where(skel)
sxy     = np.column_stack([lo[0] + (sc + 0.5) * PX, lo[1] + (sr + 0.5) * PX])
skel_rg = reg[cKDTree(pts).query(sxy)[1]]

# an edge between two 8-neighbours whose regions differ = a connection between
# two regions the 3D adjacency says do not touch
idx = {(int(c), int(r)): k for k, (c, r) in enumerate(zip(sc, sr))}
adj = [[] for _ in range(len(sxy))]
for k, (c, r) in enumerate(zip(sc, sr)):
    for dc, dr in ((1, 0), (0, 1), (1, 1), (1, -1)):
        nb = idx.get((int(c) + dc, int(r) + dr))
        if nb is not None:
            adj[k].append(nb); adj[nb].append(k)
deg  = [len(a) for a in adj]
seed = [k for k in range(len(sxy))
        if any(skel_rg[n] != skel_rg[k] for n in adj[k])]

# the flagged edge sits on a thin chain; the whole chain between the two
# junctions is what the pipeline removes, so that is what is marked here
bad, stack = np.zeros(len(sxy), bool), list(seed)
for k in seed:
    bad[k] = True
while stack:
    u = stack.pop()
    for w in adj[u]:
        if deg[w] == 2 and not bad[w]:
            bad[w] = True; stack.append(w)

extent = [lo[0], lo[0] + n_c * PX, lo[1], lo[1] + n_r * PX]

fig, ax = plt.subplots(1, 3, figsize=(11.4, 2.45))
for a in ax:
    a.set_xlim(extent[0], extent[1]); a.set_ylim(extent[2], extent[3])
    a.set_aspect("equal"); a.set_xticks([]); a.set_yticks([])
    for s in a.spines.values():
        s.set_color("0.75")

ax[0].scatter(A[:, 0], A[:, 1], s=0.25, c=C_A, lw=0, rasterized=True)
ax[0].scatter(B[:, 0], B[:, 1], s=0.25, c=C_B, lw=0, rasterized=True)
ax[0].set_title("(a) unrolled points, two regions", fontsize=10)
ax[0].annotate("gap", xy=(0.61, 0.55), xytext=(0.61, 0.20), fontsize=9,
               ha="center", color="0.25",
               arrowprops=dict(arrowstyle="->", color="0.25", lw=1))

ax[1].imshow(solid, extent=extent, origin="lower", cmap="binary", vmax=1.6,
             interpolation="nearest")
ax[1].set_title("(b) after dilation, the gap is closed", fontsize=10)

ax[2].imshow(~skel, extent=extent, origin="lower", cmap="gray", vmin=0, vmax=1,
             interpolation="nearest")
ax[2].scatter(sxy[bad, 0], sxy[bad, 1], s=9, c=C_BAD, lw=0, zorder=3)
ax[2].set_title("(c) after thinning, a strut that does not exist", fontsize=10)
ax[2].annotate("flagged", xy=(0.61, 0.55), xytext=(0.61, 0.20), fontsize=9,
               ha="center", color=C_BAD,
               arrowprops=dict(arrowstyle="->", color=C_BAD, lw=1))

fig.legend(handles=[Line2D([], [], marker="o", ls="", ms=5, color=C_A, label="region A"),
                    Line2D([], [], marker="o", ls="", ms=5, color=C_B, label="region B"),
                    Line2D([], [], color=C_BAD, lw=2,
                           label="connection between two regions that do not touch in 3D")],
           loc="lower center", ncol=3, frameon=False, fontsize=9,
           bbox_to_anchor=(0.5, 0.0))
fig.subplots_adjust(left=0.01, right=0.99, top=0.90, bottom=0.17, wspace=0.05)

fig.savefig(OUT, dpi=300, facecolor="white")
print("wrote", OUT.name, "| flagged skeleton pixels:", int(bad.sum()))

## Unrolling one ring: the halo and the seam

Figure for `fig:unroll-ring` in `03_methods.tex`, built from the real
`ring_points.csv` of the stent. One ring is opened onto the `(s, z)` plane with
`s = r_mid * theta`, exactly as `open_stent_to_plane` does it.

Drawn on top of the points:

* the **ring band**, between the two boundaries `find_rings` placed
* the **halo**, `ring_halo_frac = 0.40` of the ring height taken from each neighbour,
  so a strut crossing a boundary is not cut in the middle
* the **seam**, where the raster wraps, plus the `pad_fraction = 0.20` strips that are
  copied from the opposite side before the thinning and cropped off again afterwards

Writes `unroll_ring.png` next to this notebook.

In [ ]:
from matplotlib.lines import Line2D

# FIG_DIR and SRC come from the setup cell at the top of this notebook.

RING       = 5        # which ring is opened (1 .. 10)
HALO_FRAC  = 0.40     # ring_halo_frac in skeletonize_rings_2d
PAD_FRAC   = 0.20     # pad_fraction in compute_skeleton_2d
MAX_PTS    = 120_000  # points drawn, so the PDF stays light

C_RING = "#3a6ea5"    # the ring's own band
C_HALO = "#c98a2b"    # the halo taken from the neighbouring rings
C_SEAM = "#8c1923"    # the seam (report accent)

feat  = json.loads((SRC / "stent_features.json").read_text())
r_mid = feat["r_mid"]
edges = np.asarray(feat["ring_boundaries"], float)
circ  = 2 * np.pi * r_mid

z_lo, z_hi = edges[RING - 1], edges[RING]
halo       = HALO_FRAC * (z_hi - z_lo)
pad        = PAD_FRAC * circ

df = pd.read_csv(SRC / "ring_points.csv", usecols=["theta", "z_cylindrical"])
arc = r_mid * df["theta"].to_numpy()
z   = df["z_cylindrical"].to_numpy()

keep = (z >= z_lo - halo) & (z <= z_hi + halo)
arc, z = arc[keep], z[keep]
if len(arc) > MAX_PTS:                      # thin out for the drawing only
    sel = np.random.default_rng(0).choice(len(arc), MAX_PTS, replace=False)
    arc, z = arc[sel], z[sel]

own = (z >= z_lo) & (z <= z_hi)

fig, ax = plt.subplots(figsize=(11.0, 3.4))

# the wrap strips: a copy of each side pasted onto the other, which is what lets
# the thinning see a strut on the seam as one continuous line
for shift in (-circ, circ):
    m = np.abs(arc + shift) <= circ / 2 + pad
    ax.scatter(arc[m & own] + shift, z[m & own], s=0.25, c=C_RING, lw=0,
               alpha=0.22, rasterized=True)
    ax.scatter(arc[m & ~own] + shift, z[m & ~own], s=0.25, c=C_HALO, lw=0,
               alpha=0.22, rasterized=True)

ax.scatter(arc[own], z[own], s=0.25, c=C_RING, lw=0, rasterized=True)
ax.scatter(arc[~own], z[~own], s=0.25, c=C_HALO, lw=0, rasterized=True)

# the ring band, and the halo bands above and below it
for y in (z_lo, z_hi):
    ax.axhline(y, color="0.25", ls="--", lw=1.1, zorder=4)
ax.axhspan(z_lo - halo, z_lo, color=C_HALO, alpha=0.07, zorder=0)
ax.axhspan(z_hi, z_hi + halo, color=C_HALO, alpha=0.07, zorder=0)

# the two strips outside the seam are copies of the opposite side, added only
# so the thinning sees a strut on the seam continuously; they are cropped after
ax.axvspan(-circ / 2 - pad, -circ / 2, color="0.5", alpha=0.10, zorder=0)
ax.axvspan(circ / 2, circ / 2 + pad, color="0.5", alpha=0.10, zorder=0)

# the seam: the two vertical lines are the same line on the stent
for x in (-circ / 2, circ / 2):
    ax.axvline(x, color=C_SEAM, ls="--", lw=1.3, zorder=5)

ax.set_xlim(-circ / 2 - pad, circ / 2 + pad)
ax.set_ylim(z_lo - halo - 0.10, z_hi + halo + 0.10)
ax.set_xlabel(r"arc length $s = r_\mathrm{mid}\,\theta$  [mm]")
ax.set_ylabel(r"$z$  [mm]")

ax.annotate("seam", xy=(circ / 2, z_hi + halo), xytext=(circ / 2, z_hi + halo + 0.30),
            color=C_SEAM, fontsize=9, ha="center", va="bottom")
ax.annotate("seam", xy=(-circ / 2, z_hi + halo), xytext=(-circ / 2, z_hi + halo + 0.30),
            color=C_SEAM, fontsize=9, ha="center", va="bottom")
for sgn in (-1, 1):
    ax.text(sgn * (circ / 2 + pad / 2), z_hi + halo + 0.30, "wrap strip",
            fontsize=8, color="0.35", ha="center", va="bottom")

ax.legend(handles=[
    Line2D([], [], marker="o", ls="", ms=5, color=C_RING, label=f"ring {RING}"),
    Line2D([], [], marker="o", ls="", ms=5, color=C_HALO, label="halo from the neighbouring rings"),
    Line2D([], [], color="0.25", ls="--", lw=1.1, label="ring boundary"),
    Line2D([], [], color=C_SEAM, ls="--", lw=1.3, label="seam (both lines are the same line on the stent)"),
], loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=2, frameon=False, fontsize=9)

fig.subplots_adjust(left=0.06, right=0.995, top=0.90, bottom=0.34)
out = FIG_DIR / "methodology_figures" / "unroll_ring.png"
fig.savefig(out, dpi=300, facecolor="white")
print("wrote", out.name, "|", f"{own.sum():,} ring pts, {(~own).sum():,} halo pts",
      f"| circumference {circ:.3f} mm, band [{z_lo:.3f}, {z_hi:.3f}], halo {halo:.3f} mm")

## Stent-only results: coupling response and yield utilisation

Two figures per stent, built from the `radial_expand` and `axial_stretch`
`results/metrics.csv` written by `stentfit.sim.results`, under
`examples/data/output/simulation/stent_only/<STENT>/`.

Change `STENT` below and rerun the cell to redo both figures for a different
stent. Output is saved **into that stent's own folder**, not into the report
figures folder, so every stent's plots sit next to its own run data.

* **coupling_response.png** &mdash; the deformation that is not imposed: foreshortening
  under radial expansion, and the radial response under axial stretch.
* **yield_utilisation.png** &mdash; the bending moment at the crowns and the global peak,
  each over the first-yield moment `Mp`, against the imposed strain. Above the `M/Mp = 1`
  line a real strut would already have yielded &mdash; the elastic material here does not
  stop there, which is the whole point of the figure.

`stent01` through `stent07` all have both cases solved with
`MATERIAL_FORCES_GAUSSPOINT` on, so `crown_M_over_Mp` / `peak_M_over_Mp` exist for
all seven. `16crownCrimpedXienceStent+extremaSupports` and `2crownCrimpedXienceStent`
are **not** under this tree -- their stent-only runs live only under the older
`outputs/stent_only/` folder, which was built without that output flag, so they are
skipped here rather than plotted with missing columns.

In [ ]:
STENT = "stent01"    # change this and rerun the cell -- e.g. "stent04", "2crownCrimpedXienceStent"

SIM_ROOT = REPO / "examples" / "data" / "output" / "simulation" / "stent_only" / STENT
radial = pd.read_csv(SIM_ROOT / "radial_expand" / "results" / "metrics.csv")
axial  = pd.read_csv(SIM_ROOT / "axial_stretch" / "results" / "metrics.csv")

C_RADIAL, C_AXIAL, C_YIELD = "#3a6ea5", "#c98a2b", "#8c1923"

# Figure 1: coupling response -- the deformation that is not imposed
fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.6))
ax[0].plot(radial["radial_strain"] * 100, radial["foreshortening_pct"], "-o", ms=3, color=C_RADIAL)
ax[0].set(xlabel="imposed radial strain  [%]", ylabel="foreshortening  [%]", title="(a) radial_expand")
ax[1].plot(axial["axial_strain"] * 100, axial["radial_strain"] * 100, "-o", ms=3, color=C_AXIAL)
ax[1].set(xlabel="imposed axial strain  [%]", ylabel="radial strain response  [%]", title="(b) axial_stretch")
ax[0].grid(True, which="both", ls=":", lw=0.5, color="0.75")
ax[1].grid(True, which="both", ls=":", lw=0.5, color="0.75")
fig.suptitle("coupling response")
fig.tight_layout()
fig.savefig(FIG_DIR / "results_figures" / "coupling_response.png", dpi=300, facecolor="white")
print("wrote", FIG_DIR / "results_figures" / "coupling_response.png")
plt.show()

# Figure 2: bending moment over the first-yield moment Mp, at the junctions and at
# the single worst Gauss point anywhere in the mesh (linear scale)
radial_crown_M_over_Mp = radial["crown_M_over_Mp"]
radial_peak_M_over_Mp = radial["peak_M_over_Mp"]
axial_crown_M_over_Mp = axial["crown_M_over_Mp"]
axial_peak_M_over_Mp = axial["peak_M_over_Mp"]

fig, ax = plt.subplots(figsize=(6.2, 4.2))
ax.axhline(1, color=C_YIELD, ls="--", lw=1.2, label="M / Mp = 1, first yield")
ax.plot(radial["radial_strain"].abs() * 100, radial_crown_M_over_Mp, "-o", ms=3,
        color=C_RADIAL, label="radial_expand, junction")
ax.plot(radial["radial_strain"].abs() * 100, radial_peak_M_over_Mp, "--",
        color=C_RADIAL, alpha=0.6, label="radial_expand, peak")
ax.plot(axial["axial_strain"].abs() * 100, axial_crown_M_over_Mp, "-o", ms=3,
        color=C_AXIAL, label="axial_stretch, junction")
ax.plot(axial["axial_strain"].abs() * 100, axial_peak_M_over_Mp, "--",
        color=C_AXIAL, alpha=0.6, label="axial_stretch, peak")
ax.set(xlabel="imposed strain magnitude  [%]", ylabel="$M / M_p$",
       title=f"{STENT}: Bending Response vs. First Yield")
ax.set_yscale("log")
ax.legend(fontsize=8)
ax.grid(True, which="both", ls=":", lw=0.5, color="0.75")
fig.tight_layout()
fig.savefig(FIG_DIR / "results_figures" / "yield_utilisation.png", dpi=300, facecolor="white")
print("wrote", FIG_DIR / "results_figures" / "yield_utilisation.png")
plt.show()

## Stent-only results: the two motions on one axis

Figure for `sec:case-1` in `06_results.tex`. Both load cases move the stent along the
same axial-radial coupling, so they can be drawn on one set of axes: radial expansion
sweeps `e_r` from 0 to +60 %, axial stretch sweeps it from 0 down to -48 %. Together
they cover one continuous range through the origin.

At this scale the two branches look like one smooth curve. They are not exactly the
same, the axially driven branch starts about 2.2 times shallower than the radially
driven one, but that is a small detail and not worth a second panel to prove. If it is
worth a sentence in the text, the two numbers are printed below the figure rather than
drawn on it.

**One caveat, only relevant if that sentence gets written.** The two strains are not
measured over the same region. `radial_strain` comes from the diameter over the
central 50 % of the length, while `axial_strain` is the full length including the
gripped end bands. Under radial expansion the field is uniform so this makes no
difference, but under axial stretch the ends are clamped and the middle necks freely,
so part of the 2.2 factor may be a measurement asymmetry rather than a difference in
mechanism. Comparing `diameter` against `diameter_ends` in the axial run is the check
that separates them, the cell below prints it.


In [ ]:
# STENT, SIM_ROOT, radial, axial and the colours come from the cell above.

fig, ax = plt.subplots(figsize=(5.2, 3.9))

ax.plot(radial["radial_strain"] * 100, radial["axial_strain"] * 100, "-o", ms=3,
       color=C_RADIAL, label="radial expansion")
ax.plot(axial["radial_strain"] * 100, axial["axial_strain"] * 100, "-o", ms=3,
       color=C_AXIAL, label="axial stretch")
ax.set(xlabel="radial strain  [%]", ylabel="axial strain  [%]",
      title="the two motions on one axis")
ax.legend(fontsize=8)
ax.grid(True, ls=":", lw=0.5, color="0.75")

fig.tight_layout()
fig.savefig(SIM_ROOT / "strain_coupling.png", dpi=300, facecolor="white")
fig.savefig(FIG_DIR / "results_figures" / "strain_coupling.png", dpi=300, facecolor="white")
print("wrote", SIM_ROOT / "strain_coupling.png")
print("wrote", FIG_DIR / "results_figures" / "strain_coupling.png")
plt.show()

# the slope at the very start of each case, read directly from the first two rows
radial_slope_at_start = ((radial["axial_strain"][1] - radial["axial_strain"][0])
                        / (radial["radial_strain"][1] - radial["radial_strain"][0]))
axial_slope_at_start = ((axial["axial_strain"][1] - axial["axial_strain"][0])
                       / (axial["radial_strain"][1] - axial["radial_strain"][0]))
print("slope at the start, radial expansion:", round(radial_slope_at_start, 4))
print("slope at the start, axial stretch:   ", round(axial_slope_at_start, 4))
print("ratio:", round(radial_slope_at_start / axial_slope_at_start, 2))

# is the mid-band diameter representative in the axial case? compare it against
# the diameter measured at the ends, at full load
radial_last = radial.iloc[-1]
axial_last = axial.iloc[-1]
print()
print("mid-band vs end diameter, at full load:")
print("  radial_expand  diameter", round(radial_last["diameter"], 4), "mm   ends",
     round(radial_last["diameter_ends"], 4), "mm   difference",
     round(radial_last["stent_dogboning_pct"], 2), "%")
print("  axial_stretch  diameter", round(axial_last["diameter"], 4), "mm   ends",
     round(axial_last["diameter_ends"], 4), "mm   difference",
     round(axial_last["stent_dogboning_pct"], 2), "%")

---

# Sweeps that build the study runs

Everything above reads run folders that already exist. The cells below are what builds
them. They are kept here, next to the figures, so that a figure and the runs behind it
stay in one place.

Every sweep only builds. Solving takes hours and belongs in a terminal, so each cell
prints the commands to run at the end. The load and the step count are pinned to the
anchor run in every sweep, so the mesh is the only thing that changes.

The settings below repeat the values the report's runs were built with. They are written
out rather than taken from the class defaults, because the defaults have moved on since
those runs and the figures have to stay reproducible.

In [ ]:
from dataclasses import asdict, replace

from stentfit import Simulation, Stent
from stentfit.sim import StentBalloonSettings, StentOnlySettings

SKELETON_ROOT = REPO / "examples" / "data" / "output" / "stent_skeleton"
OUTPUT_DIR = REPO / "examples" / "data" / "output" / "simulation"
MESH_STUDY_ROOT = REPO / "examples" / "data" / "output" / "simulation_mesh_study"


def load_stent(name):
    """The skeletonised stent, by name."""
    return Stent.load(str(SKELETON_ROOT / name), stent_name=name)


def stent_only_settings(stent):
    """The stent-only settings behind `sec:case-1` and `sec:convergence`."""
    return StentOnlySettings(
        material="elastic", cases=("radial_expand", "axial_stretch"),
        youngs=2.0e5, poisson=0.3, density=0.0,
        yield_strength=300.0, tangent_modulus_ratio=64.0 / 380.0,
        beam_class="Beam3rHerm2Line3", l_el_per_strut=1.0,
        strut_thickness=stent.stent_features["strut_thickness"],
        strains={"radial_expand": 0.60, "axial_stretch": 0.10}, grip_frac=0.2,
        self_contact=False, contact_penalty_frac=0.01,
        contact_g0_per_strut=0.05, contact_exclusion_per_strut=0.5,
        n_steps=20, max_iter=20, tol_residuum=1e-8,
        predictor="TangDis", line_search="Full Step")


def stent_balloon_settings(stent):
    """The stent-balloon settings behind `sec:case-2`."""
    return StentBalloonSettings(
        material="elastic", balloon_material="orthotropic",
        load_profile="ramp_inflate_deflate",
        youngs=2.0e5, yield_strength=300.0,
        strut_thickness=stent.stent_features["strut_thickness"],
        clearance_frac=0.02, overhang_frac=0.1, wall=0.04, radial_strain=0.50,
        pressure_max=0.6, end_spring_stiffness=1000.0,
        neohooke_youngs=17.0, neohooke_poisson=0.0,
        fibre_longitudinal={"k1": 1000.0, "k2": 0.01},
        fibre_circumferential={"k1": 1.5e-7, "k2": 0.35},
        penalty=30.0, penalty_law="linear", penalty_g0_per_strut=0.5,
        discretization="gauss_point_to_segment", gauss_points=6,
        contact_type="gap_variation", mortar_shape_function="line2",
        mortar_contact_defined_in="reference_configuration",
        factor_solid=1.5, factor_beam=1.2,
        n_steps=100, max_iter=60, tol_residuum=1e-8,
        tol_increment=1e-10, predictor="ConstDis")

## Sweep: the stent-only mesh

This builds the five levels the figure below reads. There is no balloon in this case, so
the element length is set directly by `l_el_per_strut` and it is the only knob. Only
`radial_expand` is swept, and the strain and the step count stay at the production values,
so the mesh is the only thing that changes.

Each level writes into its own folder under `simulation_mesh_study` rather than into
`examples/data/output/simulation`, so the production runs behind `sec:case-1` are left
alone.

`sec:convergence` is about stent01, so that is the stent loaded here.

In [ ]:
MESH_STENT = "stent01"
SO_MESH_REFINEMENTS = [3.0, 2.0, 1.5, 1.0, 0.6]   # l_el_per_strut, coarse -> fine

mesh_stent = load_stent(MESH_STENT)
so = stent_only_settings(mesh_stent)


def build_so_mesh_case(l_el):
    """Build one stent-only radial_expand run at one mesh level, in its own folder."""
    settings = replace(so, cases=("radial_expand",), l_el_per_strut=l_el)
    output_dir = MESH_STUDY_ROOT / f"lel_{l_el:g}"
    sim = Simulation(mesh_stent, sim_type="stent_only", settings=settings,
                     output_dir=output_dir)
    sim.build_input()
    return sim.built[0], output_dir


so_mesh_study = []
for l_el in SO_MESH_REFINEMENTS:
    case, output_dir = build_so_mesh_case(l_el)
    beam = case["record"]["beam_model"]
    so_mesh_study.append((l_el, case["name"], beam["n_elements"],
                          beam["target_element_length_mm"], output_dir))

print(f"\n{'l_el_per_strut':>15s} {'run':>14s} {'beam els':>9s} {'l_el mm':>9s}")
for l_el, name, n_el, l_el_mm, _ in so_mesh_study:
    print(f"{l_el:15.2f} {name:>14s} {n_el:9d} {l_el_mm:9.4f}")

# the repo path has a space in it ("RWTH - SiSc"), so the output-dir needs quotes
# or the shell would split it into two arguments
print("\nsolve these, one terminal each:")
for l_el, name, _, _, output_dir in so_mesh_study:
    print(f'  python -m stentfit.run solve stent_only {MESH_STENT} {name} '
          f'--output-dir "{output_dir}"')

## Stent-only mesh convergence

Figure for `sec:convergence` in `06_results.tex`. Five levels of `radial_expand` on `stent01`,
built and solved by the sweep cell above, coarse to fine:
`l_el_per_strut` 3.0, 2.0, 1.5, 1.0, 0.6. The production runs use 1.0.

Foreshortening is the quantity that has to be shown converged, because it is the headline number of
`sec:case-1`. The band is a plain plus-or-minus one percentage point around the finest level, not a
fitted tolerance. If every level sits inside it, the production element length is fine.

The imposed radial strain is printed rather than plotted, because it is a check on the boundary
condition and not a result. It should read close to 0.600 at every level, since the strain is
imposed directly and does not depend on the mesh.

In [ ]:
import yaml

MESH_ROOT = REPO / "examples" / "data" / "output" / "simulation_mesh_study"
MESH_LEVELS = [3.0, 2.0, 1.5, 1.0, 0.6]   # l_el_per_strut, coarse -> fine

n_elements = []
foreshortening = []
radial_strain = []
for l_el in MESH_LEVELS:
    run_dir = MESH_ROOT / f"lel_{l_el:g}" / "stent_only" / STENT / "radial_expand"
    params = yaml.safe_load((run_dir / "run_parameters.yaml").read_text())
    summary = yaml.safe_load((run_dir / "results" / "summary.yaml").read_text())
    n_elements.append(params["beam_model"]["n_elements"])
    foreshortening.append(summary["foreshortening_pct"])
    radial_strain.append(summary["radial_strain"])

fig, ax = plt.subplots(figsize=(6.2, 4.2))
ax.plot(n_elements, foreshortening, "-o", ms=5, color=C_RADIAL)
ax.set(xlabel="beam elements", ylabel="foreshortening  [%]",
      title=f"{STENT}: mesh convergence, radial expansion")
ax.grid(True, ls=":", lw=0.5, color="0.75")

# +/- 1% around the finest level, as two dashed lines
converged = foreshortening[-1]
ax.axhline(converged - 1.0, ls="--", lw=1.2, color="0.4", label="+/- 1% of the finest level")
ax.axhline(converged + 1.0, ls="--", lw=1.2, color="0.4")
ax.legend(fontsize=8)

fig.tight_layout()
fig.savefig(FIG_DIR / "results_figures" / "mesh_convergence.png", dpi=300, facecolor="white")
print("wrote", FIG_DIR / "results_figures" / "mesh_convergence.png")
plt.show()

print("l_el_per_strut   elements   foreshortening [%]   radial strain")
for l_el, n, f, r in zip(MESH_LEVELS, n_elements, foreshortening, radial_strain):
    print(f"{l_el:14.2f}   {n:8d}   {f:17.3f}   {r:13.4f}")

## Sweep: the two stent-balloon meshes

The stent-balloon case has two meshes, so they are refined one at a time, the beams first
and then the balloon. Neither produces a figure in the report. As `sec:convergence`
explains, the coupling rules leave too little room between the two meshes to build several
distinct levels, and the scatter in the spline lengths means a finer target does not refine
every part of the beam network at the same rate. These cells are what that conclusion came
from.

The knob for the beams is `factor_beam` and not `l_el_per_strut`. In a stent-balloon run
the beam element length is tied to the balloon's element size through
`l_el = factor_beam x factor_solid x strut`, because the coupling rules constrain the ratio
between the two meshes. `l_el_per_strut` is only read by the stent-only case, where there is
no solid to couple to.

Lowering `factor_beam` refines the stent and leaves the balloon alone, which is what makes
this one variable at a time. It cannot go all the way down to 1. Both meshers round to a
whole number of elements, so the shortest beam lands below the solid size and the build
refuses it. The floor is about 1.2.

In [ ]:
BALLOON_MESH_STENT = "stent04"
MESH_PRESSURE = 0.30                     # MPa, the anchor pressure
MESH_N_STEPS  = 100                      # pinned: step count follows pressure, not mesh
BEAM_REFINEMENTS = [4.0, 2.5, 1.6, 1.2]  # factor_beam, coarse -> fine

balloon_stent = load_stent(BALLOON_MESH_STENT)
sb = stent_balloon_settings(balloon_stent)


def build_mesh_case(**overrides):
    """Build one run, returning its record. None if the coupling rules refuse it."""
    settings = replace(sb, pressure_max=MESH_PRESSURE, n_steps=MESH_N_STEPS, **overrides)
    sim = Simulation(balloon_stent, sim_type="stent_balloon", settings=settings,
                     output_dir=OUTPUT_DIR)
    try:
        sim.build_input()
    except ValueError as refused:
        print(f"  refused: {refused}")
        return None
    return sim.built[0]


beam_study = []
for factor in BEAM_REFINEMENTS:
    case = build_mesh_case(factor_beam=factor)
    if case:
        beam = case["record"]["beam_model"]
        beam_study.append((factor, case["name"], beam["n_elements"],
                           beam["target_element_length_mm"],
                           case["record"]["balloon"]["n_elements"]))

print(f"\n{'factor_beam':>12s} {'run':>8s} {'beam els':>9s} {'l_el mm':>9s} {'balloon els':>12s}")
for factor, name, n_el, l_el, n_balloon in beam_study:
    print(f"{factor:12.2f} {name:>8s} {n_el:9d} {l_el:9.4f} {n_balloon:12d}")
print("\nthe balloon element count must be identical down the last column, which is what")
print("makes this a study of the beam mesh alone.\n")

print("solve these, one terminal each:")
for _, name, _, _, _ in beam_study:
    print(f"  python -m stentfit.run solve stent_balloon {BALLOON_MESH_STENT} {name}")

The balloon is refined only once the beams have converged. `factor_solid` sets the balloon
element size as a multiple of the strut thickness, so a smaller number is a finer mesh. But
lowering it alone would refine the beams as well, since their length is derived from it, so
`factor_beam` is raised to compensate and hold the beam mesh where it was. The beam column
is the one to watch, because it must not move.

In [ ]:
# Hold l_el = factor_beam x factor_solid x strut fixed at the reference value,
# so only the balloon changes.
REFERENCE_PRODUCT = sb.factor_beam * sb.factor_solid
BALLOON_REFINEMENTS = [1.5, 1.25, 1.1, 1.0]      # factor_solid, coarse -> fine

balloon_study = []
for factor in BALLOON_REFINEMENTS:
    case = build_mesh_case(factor_solid=factor,
                           factor_beam=REFERENCE_PRODUCT / factor)
    if case:
        balloon = case["record"]["balloon"]
        balloon_study.append((factor, case["name"], balloon["n_elements"],
                              balloon["n_circumferential"],
                              case["record"]["beam_model"]["n_elements"]))

print(f"\n{'factor_solid':>13s} {'run':>8s} {'balloon els':>12s} {'around':>7s} {'beam els':>9s}")
for factor, name, n_el, n_circ, n_beam in balloon_study:
    print(f"{factor:13.2f} {name:>8s} {n_el:12d} {n_circ:7d} {n_beam:9d}")
print("\nthe beam element count must be identical down the last column.\n")

print("solve these, one terminal each:")
for _, name, _, _, _ in balloon_study:
    print(f"  python -m stentfit.run solve stent_balloon {BALLOON_MESH_STENT} {name}")

## Sweep: why stent04's dogbone does not flatten

Fig. 5 of Datz et al. shows the hump grow through mid-inflation and then disappear by
13 atm. In the run below it is still at +25% at its peak, so the two do not agree, and the
figure that follows is where that shows. There are two explanations for it, and they
predict opposite things, so one run each settles which it is.

The first is that the sprung ends are letting the balloon escape. If that is the cause,
holding the ends harder should pull the free region back. Against it, the end ring already
sits at 1.06 times its starting radius, which is what the paper's tips do as well.

The second is that the pressure is simply not high enough. The circumferential fibres
stiffen exponentially, and the hump only flattens once they reach that knee. This run sits
at a wall tension of 0.37 times the balloon modulus against the paper's 0.73, which is
close to their 6.6 atm curve, and that one still has a large hump. On this reading nothing
is wrong and the balloon has just not been pushed far enough.

The pressure variant scales `n_steps` with the pressure, so the strain per step stays what
the baseline used and the numerics do not become a second variable.

Each variant rebuilds the finished run from its own record with `Simulation.from_run`,
changes one thing, and prints the difference to show that nothing else moved.

In [ ]:
import yaml

PAPER_TENSION = 0.728       # Datz et al. at 13 atm: wall tension / balloon modulus

BASE04 = OUTPUT_DIR / "stent_balloon" / "stent04" / "run001"
base04 = Simulation.from_run(BASE04)
s04 = base04.settings
r_outer = yaml.safe_load((BASE04 / "run_parameters.yaml").read_text())["balloon"]["r_outer_mm"]

p_paper = PAPER_TENSION * s04.neohooke_youngs * s04.wall / r_outer
n_paper = round(s04.n_steps * p_paper / s04.pressure_max)   # same strain per step as the baseline

variants = {"stiffer ends":   dict(end_spring_stiffness=10.0 * s04.end_spring_stiffness),
            "paper pressure": dict(pressure_max=p_paper, n_steps=n_paper)}

print(f"baseline {BASE04.name}: {s04.pressure_max:g} MPa, {s04.n_steps} steps, "
      f"ends {s04.end_spring_stiffness:g} MPa/mm")
print(f"paper-equivalent pressure = {PAPER_TENSION} x {s04.neohooke_youngs:g} x {s04.wall:g} "
      f"/ {r_outer:.4f} = {p_paper:.4f} MPa\n")

built04 = {}
for label, overrides in variants.items():
    sim = Simulation(base04.stent, sim_type="stent_balloon",
                     settings=replace(s04, **overrides), output_dir=OUTPUT_DIR)
    sim.build_input()
    built04[label] = sim.built[0]["name"]

    before, after = asdict(s04), asdict(sim.settings)
    changed = {k: (before[k], after[k]) for k in before if before[k] != after[k]}
    print(f"{label} -> {sim.built[0]['name']}")
    for key, (old, new) in changed.items():
        print(f"    {key}: {old} -> {new}")

print("\nsolve these, one terminal each:")
for label, name in built04.items():
    print(f"  python -m stentfit.run solve stent_balloon {base04.stent.stent_name} {name}"
          f"      # {label}")

Report both afterwards and read `balloon_dogboning_pct_at_peak` out of each
`results/summary.yaml`, against the baseline's +25.0%. If the stiffer ends drop it a lot,
the end condition was the lever. If they barely move it, the springs were never the
problem. If the paper pressure drops it towards zero, the hump was a transient all along,
which is what Fig. 5 shows.

`results/balloon_profile.csv` is better evidence than the single number, because the radius
along the axis can be compared with Fig. 5 shape for shape. That is what the next cell
plots.

## Balloon dogboning: the profile and the trend

Both panels for `sec:case-2` in `06_results_and_discussion.tex`, side by side since
they are the same phenomenon looked at two ways.

Panel (a) is styled after Fig. 5 of `\citet{datz2025}`, "Balloon diameter at different
pressures for the generic free stent expansion simulation". The load profile is a
triangle that peaks at `t = 0.5` and returns to zero at `t = 1`, so the six columns of
`balloon_profile.csv` already sit at evenly spaced pressures on the inflation branch,
no new sampling is needed. The column names carry their own pressure value, so the
labels below always match whatever peak pressure this particular run used.

Panel (b) quantifies what panel (a) shows visually. `metrics.csv` carries both
`stent_dogboning_pct` and `balloon_dogboning_pct` at every load step, plotted against
pressure over the same inflation branch (`time <= 0.5`). It connects to `sec:case-1`,
where the stent's own dogboning under a uniform imposed radial field is about 0.01 %,
essentially zero, so anything seen here comes from the balloon, not from the stent's
own geometry.

Featured run: `stent04/run001`, 0.6 MPa peak, wall-tension ratio 0.741 against the
paper's 0.728.

In [ ]:
STENT = "stent04"    # change this and RUN below, then rerun the cell for another run
RUN = "run001"

SIM_ROOT = REPO / "examples" / "data" / "output" / "simulation" / "stent_balloon" / STENT / RUN
profile = pd.read_csv(SIM_ROOT / "results" / "balloon_profile.csv")
metrics = pd.read_csv(SIM_ROOT / "results" / "metrics.csv")

# the column names carry their own pressure, e.g. "r_mm_at_0.6000MPa", so the six
# labels below always match whatever peak pressure this particular run used
pressure_columns = [c for c in profile.columns if c.startswith("r_mm_at_")]

# only the inflation branch, pressure rising from 0 to its peak at t = 0.5
inflate = metrics[metrics["time"] <= 0.5]

fig, ax = plt.subplots(1, 2, figsize=(10.5, 4.2))

for col in pressure_columns:
    pressure_mpa = float(col.removeprefix("r_mm_at_").removesuffix("MPa"))
    ax[0].plot(profile["z_mm"], 2 * profile[col], label=f"{pressure_mpa:.2f} MPa")
ax[0].set(xlabel="axial position z  [mm]", ylabel="balloon diameter  [mm]",
          title="(a) balloon diameter at different pressures")
ax[0].legend(fontsize=7)
ax[0].grid(True, ls=":", lw=0.5, color="0.75")

ax[1].plot(inflate["pressure_MPa"], inflate["stent_dogboning_pct"], "-o", ms=3, label="stent")
ax[1].plot(inflate["pressure_MPa"], inflate["balloon_dogboning_pct"], "-o", ms=3, label="balloon")
ax[1].set(xlabel="pressure  [MPa]", ylabel="dogboning  [%]",
          title="(b) dogboning against pressure")
ax[1].legend(fontsize=8)
ax[1].grid(True, ls=":", lw=0.5, color="0.75")

fig.suptitle("dogboning")
fig.tight_layout()
fig.savefig(SIM_ROOT / "results" / "dogboning_profile.png", dpi=300, facecolor="white")
fig.savefig(FIG_DIR / "results_figures" / "dogboning_profile.png", dpi=300, facecolor="white")
print("wrote", SIM_ROOT / "results" / "dogboning_profile.png")
print("wrote", FIG_DIR / "results_figures" / "dogboning_profile.png")
plt.show()

## Sweep: is the contact tight enough?

Penalty contact is a spring and not a wall. The pushing force is the overlap times the
penalty parameter, so some overlap always exists and it grows with the load. On stent01 the
struts sink 6% of a strut diameter into the balloon at 0.15 MPa and 44% at 0.60 MPa. The
figure below is what that looks like over a whole run.

That is an accuracy limit of the formulation rather than an error, and there is one way to
find out whether it changes the answer. The same case is solved again with a stiffer
contact, and the diameter is compared.

The hard part is the step where the balloon first touches the stent. At the start of that
step the gap is still positive, so the tangent carries no contact stiffness at all. Newton
then predicts a free advance of a whole step, which is about 0.019 mm, and lands well inside
the strut. The restoring force is the penalty times that overshoot, and it is what killed the
first two attempts, at a penalty of 100 and of 30, both at exactly that step.

The overshoot is the balloon's advance per step, so the fix is a finer approach. The step
rule holds the pressure per step constant, which is why every run so far reaches contact at
t = 0.035, and lowering the pressure alone changes nothing. This cell therefore uses four
times the step count the rule asks for, which is only affordable at the low end of the
sweep. That is why run003 is the baseline here rather than run005.

In [ ]:
BASELINE = OUTPUT_DIR / "stent_balloon" / "stent01" / "run003"   # 0.15 MPa

PENALTY_TEST = 30.0         # N/mm^2, three times the paper's value
STEP_FACTOR  = 4            # finer than the rule asks, to soften the first-contact overshoot

base = Simulation.from_run(BASELINE)        # stent and output folder come from the record too
test = Simulation(base.stent, sim_type="stent_balloon",
                  settings=replace(base.settings,
                                   penalty=PENALTY_TEST,
                                   n_steps=STEP_FACTOR * base.settings.n_steps),
                  output_dir=OUTPUT_DIR)
test.build_input()
name = test.built[0]["name"]

before, after = asdict(base.settings), asdict(test.settings)
changed = {k: (before[k], after[k]) for k in before if before[k] != after[k]}
print(f"\n{BASELINE.name} -> {name}, differences:")
for key, (old, new) in changed.items():
    print(f"  {key}: {old} -> {new}")
if set(changed) != {"penalty", "n_steps"}:
    print("  !! expected penalty and n_steps only -- do not solve this until it is")

approach = 0.019 * base.settings.n_steps / test.settings.n_steps
print(f"\n  approach at first contact ~{approach:.4f} mm/step, "
      f"onset force ~{PENALTY_TEST * approach * 0.75:.2f} "
      f"(run005 survived ~0.14, run007 died at ~0.42)")
print(f"\n  python -m stentfit.run solve stent_balloon {base.stent.stent_name} {name}")
print(f"  (built in {test.built[0]['run_dir']})")

Report it afterwards and compare against run003. Its `penetration_per_strut_at_peak` is
0.059, so the stiffer contact should come out at roughly a third of that, and its
`diameter_peak_mm` is 3.3126, which is the number that decides the question. If the diameter
moves by well under a percent, the contact is tight enough and the sweep stands.

One thing has to be stated if this goes into the report. The pair differs in step count as
well as in penalty, because the finer approach is what makes the stiffer contact solvable at
all. A quasi-static elastic run should not depend on the step count, and the four runs of the
sweep span 50 to 200 steps on a smooth compliance curve, which supports that. But it is an
assumption here and not a measurement.

If it diverges anyway, the log will show it at a low step number with the gap crossing zero,
and not at the peak.

## Contact behaviour against pressure

Figure for `sec:case-2` in `06_results_and_discussion.tex`. `metrics.csv` carries
`penetration_per_strut` and `n_nodes_in_contact` at every load step. Penalty contact
is a stiff spring rather than a hard constraint, so some overlap always exists and
grows with load, this is stated as an accuracy limit of the formulation rather than
an error.

Only the inflation branch (`time <= 0.5`), same as the dogboning figure.

In [ ]:
STENT = "stent04"    # stent04, stent01, 2crownCrimpedXienceStent
RUN = "run001"

SIM_ROOT = REPO / "examples" / "data" / "output" / "simulation" / "stent_balloon" / STENT / RUN
metrics = pd.read_csv(SIM_ROOT / "results" / "metrics.csv")

# only the inflation branch, pressure rising from 0 to its peak at t = 0.5
inflate = metrics[metrics["time"] <= 0.5]

fig, ax = plt.subplots(1, 2, figsize=(10.5, 4.2))

ax[0].plot(inflate["pressure_MPa"], inflate["penetration_per_strut"] * 100, "-o", ms=3, color="#3a6ea5")
ax[0].set(xlabel="pressure  [MPa]", ylabel="penetration  [% of strut diameter]",
          title="(a) contact penetration against pressure")
ax[0].grid(True, ls=":", lw=0.5, color="0.75")

ax[1].plot(inflate["pressure_MPa"], inflate["n_nodes_in_contact"], "-o", ms=3, color="#c98a2b")
ax[1].set(xlabel="pressure  [MPa]", ylabel="nodes in contact",
          title="(b) nodes in contact against pressure")
ax[1].grid(True, ls=":", lw=0.5, color="0.75")

fig.suptitle("contact behaviour")
fig.tight_layout()
fig.savefig(SIM_ROOT / "results" / "contact_behaviour.png", dpi=300, facecolor="white")
fig.savefig(FIG_DIR / "results_figures" / "contact_behaviour.png", dpi=300, facecolor="white")
print("wrote", SIM_ROOT / "results" / "contact_behaviour.png")
print("wrote", FIG_DIR / "results_figures" / "contact_behaviour.png")
plt.show()